# 패턴 비교와 선택

같은 질문·문서·정답·모델로 기준선, Multi-Query, HyDE를 비교합니다.

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** 패턴 비교는 질문·문서·정답 ID·모델·상위 k를 고정하고 바꾸려는 검색 패턴만 달리합니다. 각 행에는 검색 후보, 정답 순위, 생성 사용량, 지연을 같은 열로 기록해 질문별 차이와 전체 요약을 함께 읽습니다.

**다른 상황에 적용.** 새로운 고객지원 데이터에서도 기준선, Multi-Query, HyDE를 같은 실패 질문 묶음에 적용할 수 있습니다. 평균만 보지 말고 어떤 질문 유형에서 순위가 올랐고 다른 질문에서 내려갔는지 사례별로 확인합니다.

**비교와 경계.** 패턴 선택은 기능 목록을 고르는 일이 아니라 실패 원인에 맞는 개입을 고르는 일입니다. 표현 불일치에는 Multi-Query, 복합 요구에는 분해, 배경 규칙 부족에는 Step-back이 후보입니다. 이 실험의 세 질문 결과를 전체 서비스 성능으로 일반화할 수 없습니다.


## 실행 환경 준비

고정한 Python 패키지를 확인합니다.

**이 셀의 코드 읽기.** `%pip install`은 이 노트북 커널이 사용할 패키지를 설치합니다. 앞 셀의 값을 입력으로 사용하지 않으며, 설치가 끝난 뒤의 코드 셀에서 패키지를 가져와 사용합니다.


In [1]:
%pip install -q langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3 numpy==2.5.3
import sys
print("준비 완료:", sys.version.split()[0])

Note: you may need to restart the kernel to use updated packages.
준비 완료: 3.12.13


## .env 찾기

가까운 .env에서 키를 읽고 존재 여부만 표시합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [2]:
# 현재 폴더에서 가까운 .env를 찾되 비밀 값은 출력하지 않습니다.
import os, time
from dotenv import find_dotenv, load_dotenv
env_path=find_dotenv(usecwd=True)
load_dotenv(env_path)
print("OPENAI_API_KEY:","있음" if os.getenv("OPENAI_API_KEY") else "없음")

OPENAI_API_KEY: 있음


## 공통 데이터

Ch12의 축소 데이터와 Ch11 방식의 정답 문서 ID를 사용합니다.

**이 셀의 코드 읽기.** `json.loads(text)`는 JSON 문자열을 파이썬 딕셔너리나 목록으로 변환해 반환합니다. `read_text()`는 파일 내용을 문자열로 반환합니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [3]:
# 배포 묶음에서도 같은 상대 경로를 찾도록 프로젝트 루트를 거슬러 올라갑니다.
import json
from pathlib import Path
import numpy as np
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"labs").is_dir())
data_dir=root/"labs/part4/ch14-rag-patterns/data"
docs=json.loads((data_dir/"documents.json").read_text(encoding="utf-8"))
queries=json.loads((data_dir/"queries.json").read_text(encoding="utf-8"))
print("문서·질문:",len(docs),len(queries))

문서·질문: 12 8


## 모델과 임베딩

모델과 256차원 문서 벡터를 준비합니다.

**이 셀의 코드 읽기.** `OpenAIEmbeddings(...)`는 문자열을 수치 벡터로 변환할 임베딩 클라이언트를 준비합니다. 실제 호출은 `embed_query()`나 `embed_documents()`에서 일어납니다. `embed_documents(texts)`는 여러 문장을 입력받아 문장 수만큼의 벡터를 반환합니다. `init_chat_model(...)`은 채팅 모델 객체를 만들며, 이 줄에서는 아직 답변을 생성하지 않습니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [4]:
# 256차원은 Ch12와 같은 비교 조건이며 행 하나가 문서 하나입니다.
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
chat=init_chat_model("gpt-4.1-mini",model_provider="openai",temperature=0)
embedder=OpenAIEmbeddings(model="text-embedding-3-small",dimensions=256)
vectors=np.array(embedder.embed_documents([d["text"] for d in docs]))
print("임베딩 배열:",vectors.shape)

임베딩 배열: (12, 256)


## 검색·생성 함수

검색 점수, 토큰, 시간, 추정 비용을 같은 방식으로 기록합니다.

**이 셀의 코드 읽기.** `search(question: str, k: int = 3)`, `ask(prompt: str)`, `show_hits(hits: list[tuple[dict[str, Any], float]])`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `invoke(input)`는 정의한 함수를 뒤에서 호출할 때 해당 함수 본문 안에서 모델이나 그래프를 실행하고 응답 객체를 반환합니다. `embed_query(text)`는 검색 질문 한 개를 벡터 하나로 반환합니다.


In [5]:
# 코사인 유사도는 내적을 두 벡터의 길이로 나누어 계산합니다.
from typing import Any
usage_log: list[dict[str, float | int]] = []

def search(question: str, k: int = 3) -> list[tuple[dict[str, Any], float]]:
    q=np.array(embedder.embed_query(question))
    scores=vectors@q/(np.linalg.norm(vectors,axis=1)*np.linalg.norm(q))
    # argsort 결과를 뒤집어 점수가 큰 문서부터 k개를 고릅니다.
    return [(docs[i],float(scores[i])) for i in np.argsort(scores)[::-1][:k]]
def ask(prompt: str) -> tuple[str, int, float]:
    started=time.perf_counter()
    msg=chat.invoke(prompt)
    usage=msg.usage_metadata or {}
    details=usage.get("input_token_details", {})
    input_tokens=usage.get("input_tokens", 0)
    cached_tokens=details.get("cache_read", 0)
    output_tokens=usage.get("output_tokens", 0)
    tokens=usage.get("total_tokens", 0)
    # 2026-09-10 gpt-4.1-mini 공개 단가로 추정하며 실제 청구액과 다를 수 있습니다.
    usd=((input_tokens-cached_tokens)*0.40+cached_tokens*0.10+output_tokens*1.60)/1_000_000
    usage_log.append({"input":input_tokens,"cached":cached_tokens,"output":output_tokens,"usd":usd})
    return msg.text,tokens,round((time.perf_counter()-started)*1000,1)

def show_hits(hits: list[tuple[dict[str, Any], float]]) -> list[tuple[str, float]]:
    return [(d["doc_id"],round(score,3)) for d,score in hits]

## 비교 함수

질문 한 건과 패턴 이름을 입력받아 검색부터 답 생성까지 실행합니다. 반환할 후보·정답 순위·사용량 필드를 확인합니다.

**이 셀의 코드 읽기.** `run_variant(item: dict, variant: str)`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다.


In [6]:
# 패턴별 검색·답변·사용량을 같은 필드로 반환합니다.
def run_variant(item: dict, variant: str) -> dict:
    started=time.perf_counter()
    usage_start=len(usage_log)
    if variant=="baseline":
        hits=search(item["question"],3)
    elif variant=="multi":
        text,_,_=ask(f"검색 질문 2개를 줄마다 쓰세요:{item['question']}")
        qs=[x.lstrip("-12. ") for x in text.splitlines() if x.strip()][:2]
        rrf={}
        found={}
        for q in qs:
            for rank,(doc,score) in enumerate(search(q,3),1):
                found[doc["doc_id"]]=(doc,score)
                rrf[doc["doc_id"]]=rrf.get(doc["doc_id"],0)+1/(60+rank)
        ids=sorted(rrf,key=rrf.get,reverse=True)[:3]
        hits=[found[doc_id] for doc_id in ids]
    else:
        text,_,_=ask(f"가상 안내문 2문장:{item['question']}")
        hits=search(text,3)
    context="\n".join(d["text"] for d,_ in hits)
    answer,_,_=ask(f"근거로만 짧게 답하세요.\n{context}\n질문:{item['question']}")
    used=usage_log[usage_start:]
    ids=[d["doc_id"] for d,_ in hits]
    return {"pattern":variant,"qid":item["qid"],"question":item["question"],"gold":item["gold_doc_id"],"retrieved_ids":ids,"answer":answer,"rank":next((i for i,x in enumerate(ids,1) if x==item["gold_doc_id"]),None),"input_tokens":sum(x["input"] for x in used),"cached_tokens":sum(x["cached"] for x in used),"output_tokens":sum(x["output"] for x in used),"usd":sum(x["usd"] for x in used),"ms":round((time.perf_counter()-started)*1000,1)}

## 세 패턴 실행

같은 세 질문을 기준선·Multi-Query·HyDE로 실행합니다. 질문별 검색 후보와 답, 전체 지연 시간을 행 단위로 확인합니다.

**이 셀의 코드 읽기.** `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [7]:
# 같은 세 질문을 기준선, Multi-Query, HyDE에 차례로 실행합니다.
sample=[queries[i] for i in [1,3,7]]
rows=[]
for variant in ["baseline","multi","hyde"]:
    for item in sample:
        rows.append(run_variant(item,variant))
for row in rows:
    print(row)

{'pattern': 'baseline', 'qid': 'C02', 'question': '책을 늦게 반납하면 돈을 내나요?', 'gold': 'LIB-002', 'retrieved_ids': ['LIB-004', 'LIB-003', 'LIB-001'], 'answer': '본문에 반납 지연 시 벌금에 대한 언급이 없습니다.', 'rank': None, 'input_tokens': 108, 'cached_tokens': 0, 'output_tokens': 16, 'usd': 6.88e-05, 'ms': 974.5}
{'pattern': 'baseline', 'qid': 'C04', 'question': '스터디룸 노쇼 취소 기준은 몇 분인가요?', 'gold': 'FAC-001', 'retrieved_ids': ['FAC-001', 'FAC-004', 'LIB-001'], 'answer': '예약 시작 10분 뒤까지 입실하지 않으면 취소됩니다.', 'rank': 1, 'input_tokens': 123, 'cached_tokens': 0, 'output_tokens': 16, 'usd': 7.480000000000002e-05, 'ms': 842.6}
{'pattern': 'baseline', 'qid': 'C08', 'question': '분실한 카드 사용을 막으려면?', 'gold': 'ACC-003', 'retrieved_ids': ['ACC-003', 'ACC-001', 'FAC-004'], 'answer': '포털에서 즉시 사용 정지를 신청합니다.', 'rank': 1, 'input_tokens': 113, 'cached_tokens': 0, 'output_tokens': 11, 'usd': 6.280000000000001e-05, 'ms': 899.1}
{'pattern': 'multi', 'qid': 'C02', 'question': '책을 늦게 반납하면 돈을 내나요?', 'gold': 'LIB-002', 'retrieved_ids': ['LIB-

## Recall·토큰·시간 집계

패턴별 세 행을 모아 Recall@3과 평균 토큰·시간·추정 비용을 계산합니다. 한 지표만 좋아진 경우도 그대로 비교합니다.

**이 셀의 코드 읽기.** `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [8]:
# 패턴마다 정답 포함률과 전체 호출 비용을 평균·합계로 계산합니다.
for variant in ["baseline","multi","hyde"]:
    part=[r for r in rows if r["pattern"]==variant]
    recall=sum(r["rank"] is not None for r in part)/len(part)
    avg_tokens=sum(r["input_tokens"]+r["output_tokens"] for r in part)/len(part)
    avg_ms=sum(r["ms"] for r in part)/len(part)
    total_usd=sum(r["usd"] for r in part)
    print(variant,{"Recall@3":round(recall,3),"평균_토큰":round(avg_tokens,1),"평균_ms":round(avg_ms,1),"USD":round(total_usd,6)})

baseline {'Recall@3': 0.667, '평균_토큰': 129.0, '평균_ms': 905.4, 'USD': 0.000206}
multi {'Recall@3': 0.667, '평균_토큰': 188.0, '평균_ms': 1863.2, 'USD': 0.000379}
hyde {'Recall@3': 1.0, '평균_토큰': 197.7, '평균_ms': 1945.0, 'USD': 0.000414}


## 공개 실행 로그 저장

문서·질문 해시와 아홉 실행 행을 JSON으로 저장합니다. 다음 챕터가 같은 입력과 결과를 추적할 필드가 모두 있는지 확인합니다.

**이 셀의 코드 읽기.** `write_text(text)`가 실행되는 줄에서 문자열이 실제 파일에 저장됩니다. `sha256(bytes)`는 같은 입력인지 비교할 해시 객체를 만들며 원문을 복원하는 암호화 함수가 아닙니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [9]:
# 데이터 해시와 질문별 원자료를 JSON 실행 로그로 저장합니다.
import hashlib
document_hash=hashlib.sha256((data_dir/"documents.json").read_bytes()).hexdigest()
query_hash=hashlib.sha256((data_dir/"queries.json").read_bytes()).hexdigest()
log={"source_provenance":"Ch12 core 축소 문서 12건·질문 8건","chat_model":"gpt-4.1-mini","embedding_model":"text-embedding-3-small","embedding_dimensions":256,"document_hash":document_hash,"query_hash":query_hash,"rows":rows}
runs=data_dir.parent/"runs"
runs.mkdir(exist_ok=True)
path=runs/"pattern-comparison.json"
path.write_text(json.dumps(log,ensure_ascii=False,indent=2),encoding="utf-8")
print("저장:",path.relative_to(root),"행:",len(rows))

저장: labs/part4/ch14-rag-patterns/runs/pattern-comparison.json 행: 9
